In [1]:
import sys
from ortools.constraint_solver import pywrapcp
from ortools.constraint_solver import routing_enums_pb2

In [2]:
def read_input():
    data = sys.stdin.readline
    n, k = map(int, data().split())
    c = list(map(int, data().split()))
    d = [0] * (n + 1)
    d[1:] = list(map(int, data().split()))
    dist = []
    for i in range(n + 1):
        dist.append(list(map(int, data().split())))
    return n, k, c, d, dist

In [6]:
def solve(n, k, c, d, dist):
    manager = pywrapcp.RoutingIndexManager(n + 1, k, 0)
    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return dist[from_node][to_node]

    transit_distance_callback = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_distance_callback)

    def capacity_callback(from_index):
        return d[manager.IndexToNode(from_index)]

    capacity_callback_index = routing.RegisterUnaryTransitCallback(capacity_callback)
    routing.AddDimensionWithVehicleCapacity(
        capacity_callback_index,
        0,
        c,
        True,
        'Capacity'
    )

    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PARALLEL_CHEAPEST_INSERTION
    search_parameters.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    search_parameters.time_limit.FromSeconds(5)
    solution = routing.SolveWithParameters(search_parameters)

    status = routing.status()
    if status == -1:
        print(-1)
        return
    
    if solution:
        print(solution.ObjectiveValue())
        for vehicle_id in range(k):
            index = routing.Start(vehicle_id)
            route = []
            route_distance = 0

            while not routing.IsEnd(index):
                node_index = manager.IndexToNode(index)
                route.append(node_index)
                previous_index = index
                index = solution.Value(routing.NextVar(index))
                route_distance += routing.GetArcCostForVehicle(previous_index, index, vehicle_id)

            route.append(manager.IndexToNode(index))
            if len(route) > 2:
                print(f"Lộ trình của xe thứ {vehicle_id + 1}:", *route)
                print(f"Quãng đường của xe thứ {vehicle_id + 1}:", route_distance)

In [7]:
def main():
    f = open("input.txt", "r")
    sys.stdin = f
    n, k, c, d, dist = read_input()
    solve(n, k, c, d, dist)
    f.close()

In [8]:
main()

81
Lộ trình của xe thứ 1: 0 2 1 0
Quãng đường của xe thứ 1: 35
Lộ trình của xe thứ 2: 0 3 4 5 0
Quãng đường của xe thứ 2: 46
